In [1]:
from torch_geometric.datasets import KarateClub
import yaml

dataset = KarateClub()
print(f'Dataset: {dataset}:')
print('======================')
print(f'Number of graphs: {len(dataset)}')
print(f'Number of features: {dataset.num_features}')
print(f'Number of classes: {dataset.num_classes}')

data = dataset[0]  # Get the first graph object.

c:\Users\KETI\anaconda3\envs\ai_tutorial\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset: KarateClub():
Number of graphs: 1
Number of features: 34
Number of classes: 4


In [2]:
# Load best config from wandb sweep
# path = "./wandb/sweep-df5mbgsm/config-cqvha3fh.yaml" # (GCN) put best config path here 
# path = "./wandb/sweep-2sl6k1ms/config-eouqxlrp.yaml" # (GAT) put best config path here
# path = "./wandb/sweep-utsldekv/config-uh7mds6z.yaml" # (GraphSAGE) put best config path here
path = "./wandb/sweep-vllbse97/config-t3yov5ho.yaml" # (DNN) put best config path here
with open(path, "r", encoding="utf-8") as f:
    raw_config = yaml.safe_load(f)

# W&B YAML stores each parameter as {value: ...}.
best_config = {
    key: entry["value"]
    for key, entry in raw_config.items()
    if isinstance(entry, dict) and "value" in entry
}

def config_kwargs(prefix):
    return {
        key[len(prefix):]: value # round(value, 3)
        for key, value in best_config.items()
        if key.startswith(prefix)
    }

In [3]:
best_config

{'models.DNN.dropout': 0.2416828984561134,
 'models.DNN.hidden_channels': 64,
 'models.DNN.num_layers': 2,
 'optimizer.Adam.lr': 0.0028141549517563137,
 'optimizer.Adam.weight_decay': 0.0007791373130045583,
 'training.gradient_clip_val': 1.0062723345005873,
 'wandb_project': 'karate-club-gcn-sweep'}

In [4]:
import torch
import torch.nn.functional as F

from utils.model import GCN, GAT, GraphSAGE, DNN

import random

def seed_everything(seed):
    """ Set random seeds for reproducibility.
    Args:
        seed (int): Random seed.
    """
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)
# model_kwargs = config_kwargs("models.GCN.")
# model = GCN(in_channels=dataset.num_features,
#             num_classes=dataset.num_classes,
#             **model_kwargs)
# model_kwargs = config_kwargs("models.GAT.")
# model = GAT(in_channels=dataset.num_features,
#             num_classes=dataset.num_classes,
#             **model_kwargs)
# model_kwargs = config_kwargs("models.GraphSAGE.")
# model = GraphSAGE(in_channels=dataset.num_features,
#             num_classes=dataset.num_classes,
#             **model_kwargs)
model_kwargs = config_kwargs("models.DNN.")
model = DNN(in_channels=dataset.num_features,
            num_classes=dataset.num_classes,
            **model_kwargs)

print("Model config:", model_kwargs)
print(model)

Model config: {'dropout': 0.2416828984561134, 'hidden_channels': 64, 'num_layers': 2}
DNN(
  (fcs): ModuleList(
    (0): Linear(in_features=34, out_features=64, bias=True)
    (1): Linear(in_features=64, out_features=64, bias=True)
  )
  (classifier): Linear(in_features=64, out_features=4, bias=True)
)


In [5]:
criterion = torch.nn.CrossEntropyLoss()  # Define loss criterion.
optimizer_kwargs = config_kwargs("optimizer.Adam.")
# PyYAML may parse scientific notation as a string.
optimizer_kwargs = {key: float(value) for key, value in optimizer_kwargs.items()}
optimizer = torch.optim.Adam(model.parameters(), **optimizer_kwargs)
gradient_clip_val = float(best_config["training.gradient_clip_val"])
print("Adam config:", optimizer_kwargs)
print("Gradient clip:", gradient_clip_val)

def train(data):
    model.train()
    optimizer.zero_grad()  # Clear gradients.
    out = model(data.x, data.edge_index)  # Perform a single forward pass.
    loss = criterion(out[data.train_mask], data.y[data.train_mask])  # Compute the loss solely based on the training nodes.
    loss.backward()  # Derive gradients.
    if gradient_clip_val > 0:
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=gradient_clip_val)
    optimizer.step()  # Update parameters based on gradients.
    return loss

loss_history = []
for epoch in range(500):
    loss = train(data)
    loss_history.append(loss.item())
    if epoch % 10 == 0:
        print(f"epoch={epoch}, loss={loss.item():.4f}")

Adam config: {'lr': 0.0028141549517563137, 'weight_decay': 0.0007791373130045583}
Gradient clip: 1.0062723345005873
epoch=0, loss=1.3862
epoch=10, loss=1.3457
epoch=20, loss=1.2598
epoch=30, loss=0.9843
epoch=40, loss=0.6324
epoch=50, loss=0.2945
epoch=60, loss=0.1304
epoch=70, loss=0.1029
epoch=80, loss=0.0413
epoch=90, loss=0.0203
epoch=100, loss=0.0120
epoch=110, loss=0.0056
epoch=120, loss=0.0021
epoch=130, loss=0.0437
epoch=140, loss=0.0032
epoch=150, loss=0.0036
epoch=160, loss=0.0055
epoch=170, loss=0.0183
epoch=180, loss=0.0026
epoch=190, loss=0.0016
epoch=200, loss=0.0005
epoch=210, loss=0.0029
epoch=220, loss=0.0005
epoch=230, loss=0.0065
epoch=240, loss=0.0040
epoch=250, loss=0.0099
epoch=260, loss=0.0013
epoch=270, loss=0.0010
epoch=280, loss=0.0217
epoch=290, loss=0.0005
epoch=300, loss=0.0030
epoch=310, loss=0.0011
epoch=320, loss=0.0033
epoch=330, loss=0.0008
epoch=340, loss=0.0008
epoch=350, loss=0.0026
epoch=360, loss=0.0035
epoch=370, loss=0.0072
epoch=380, loss=0.000

In [ ]:
# import numpy as np

# np.save("./loss_curve_vis/data/dnn_train_loss_hpt.npy", np.array(loss_history))

In [6]:
model.eval()
logits = model(data.x, data.edge_index)
test_mask = ~data.train_mask
test_loss = F.cross_entropy(logits[test_mask], data.y[test_mask]).item()
test_accuracy = (
        F.softmax(logits[test_mask],dim=1).argmax(dim=1) == data.y[test_mask]
    ).float().mean().item()

print("test loss: ", test_loss)
print("test accuracy: ", test_accuracy)

test loss:  1.3821841478347778
test accuracy:  0.36666667461395264


In [8]:
# 예측 값
F.softmax(logits[test_mask],dim=1).argmax(dim=1)

tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1])

In [9]:
# 실제 값
data.y[test_mask]

tensor([1, 1, 1, 3, 3, 1, 1, 3, 1, 1, 1, 0, 0, 3, 1, 0, 1, 0, 1, 0, 0, 2, 0, 0,
        2, 0, 0, 2, 0, 0])